### Determining the optimal number of neurons and layers for the ANN 
###### This can be done using techniques like Grid Search or Random Search to find the best combination of hyperparameters that yield the highest accuracy on the validation set. 
###### Begin with a simple ANN architecture and then systematically vary the number of neurons and layers to evaluate their impact on model performance. 
###### Use grid search to automate the process of hyperparameter tuning and identify the best configuration for the ANN. 
###### Heuristic methods can also be applied to narrow down the search space for hyperparameters, such as starting with a small number of neurons and layers and gradually increasing them based on performance metrics.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [2]:
df = pd.read_csv('Churn_Modelling.csv')

In [3]:
df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1, inplace=True)

label_encoder = LabelEncoder()
df['Gender'] = label_encoder.fit_transform(df['Gender'])
onehot_encoder = OneHotEncoder(handle_unknown='ignore')
geography_encoded = onehot_encoder.fit_transform(df[['Geography']]).toarray()
geography_df = pd.DataFrame(geography_encoded, columns=onehot_encoder.get_feature_names_out(['Geography']))
df = pd.concat([df, geography_df], axis=1)
df.drop('Geography', axis=1, inplace=True)

X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

with open('onehot_encoder.pkl', 'wb') as f:
    pickle.dump(onehot_encoder, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [4]:
def create_model(neurons=32, layers=1, activation='relu', optimizer='adam'):
    model = Sequential()

    # Input layer
    model.add(Dense(neurons, input_shape=(X_train.shape[1],), activation=activation))

    # Hidden layers
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation=activation))

    # Output layer
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        loss='binary_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy']
    )

    return model

In [5]:
model = KerasClassifier(
    model=create_model,
    verbose=0
)

In [6]:
##Define the grid search parameters for hyperparameter tuning
param_grid = {
    "model__neurons": [16, 32, 64],
    "model__layers": [1, 2, 3],
    "epochs": [50, 100],
    "batch_size": [32]
}

In [8]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)

In [10]:
print("Best Parameters:", grid_result.best_params_)
print("Best Accuracy:", grid_result.best_score_)

Best Parameters: {'batch_size': 32, 'epochs': 50, 'model__layers': 2, 'model__neurons': 16}
Best Accuracy: 0.8567490110247847


In [11]:
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))

Best: 0.856749 using {'batch_size': 32, 'epochs': 50, 'model__layers': 2, 'model__neurons': 16}
